# ANEXO F — Dominio de aplicabilidad Mw × Rrup — reconstrucción reproducible

**Tesis:** Desarrollo y validación de modelos de aprendizaje automático para la estimación de Sa(T=1.0 s) en sismos de interfaz de subducción de la placa de Nazca.

**Estado:** `RECONSTRUIDO PARA REPRODUCIBILIDAD`.

Este notebook reconstruye el análisis posterior al congelamiento descrito en la Sección 3.9.1 y reportado en la Sección 4.7 de la tesis. **No es un notebook histórico original** de la corrida V5.1. Parte exclusivamente de artefactos oficiales ya congelados: `01_dataset_analitico_v5_1987.csv` y `05_hiperparametros_finales.json`.

El notebook maestro histórico que sustenta V5.1 tiene SHA-256:

`f55fa09a43c3811984d93278ee30f5d80a05de1a421ef8eded33f8a87e4036c5`

No se modifican algoritmo, variables ni hiperparámetros. La finalidad es volver a generar de forma auditable las predicciones LOEO por registro, la matriz 8×8 Mw–Rrup, la clasificación operativa y el diagnóstico específico de la ventana de Lima.

**Revisión de auditoría v2:** corrige únicamente trazabilidad/rutas, hashes y controles de integridad. No modifica V5.1, sus variables, hiperparámetros ni reglas de clasificación.


## 0. Entorno reproducible

La corrida oficial V5.1 registró: Python 3.13.15, pandas 2.2.3, NumPy 2.1.3 y scikit-learn 1.6.1. Para comparar cifras a cuatro decimales debe utilizarse ese entorno. Si las versiones no coinciden, el notebook se detiene antes del modelamiento para evitar producir una matriz aparentemente “oficial” con otra implementación.

In [ ]:
import os, json, platform, hashlib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    print('No se detectó Google Colab; se continúa con el sistema de archivos disponible.')

EXPECTED = {
    'python': '3.13.15',
    'pandas': '2.2.3',
    'numpy': '2.1.3',
    'sklearn': '1.6.1'
}
ACTUAL = {
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__
}
print('Esperado:', EXPECTED)
print('Actual:  ', ACTUAL)

if ACTUAL != EXPECTED:
    raise RuntimeError(
        'ENTORNO NO OFICIAL. Para reproducibilidad estricta use las versiones registradas en V5.1. '
        'No continúe con la generación de resultados oficiales hasta igualarlas.'
    )

### Nota para recrear el entorno
En un runtime limpio puede ser necesario instalar explícitamente las versiones. La instalación y reinicio del kernel deben realizarse **antes** de ejecutar el resto del notebook. Por ejemplo:

`pip install pandas==2.2.3 numpy==2.1.3 scikit-learn==1.6.1`

La disponibilidad de esas versiones depende del repositorio/entorno utilizado.

In [ ]:
RUTA_DATOS = '/content/drive/MyDrive/Tesis/Datos'
RUTA_RESULTADOS = os.path.join(RUTA_DATOS, 'Resultados_V5')

# Repositorio OFICIAL de anexos:
RUTA_ANEXOS_OFICIAL = '/content/drive/MyDrive/Tesis/Tesis_Vigente/Anexos'
RUTA_ANEXO = os.path.join(RUTA_ANEXOS_OFICIAL, 'F_Dominio_Aplicabilidad_Lima_E030')
os.makedirs(RUTA_ANEXO, exist_ok=True)

# Se priorizan las copias oficiales de los Anexos B y C.
# Se conserva fallback a Resultados_V5 para reproducir el árbol histórico de trabajo.
DATASET_CANDIDATOS = [
    os.path.join(RUTA_ANEXOS_OFICIAL, 'B_Trazabilidad_Datos', '01_dataset_analitico_v5_1987.csv'),
    os.path.join(RUTA_RESULTADOS, '01_dataset_analitico_v5_1987.csv'),
]
PARAMS_CANDIDATOS = [
    os.path.join(RUTA_ANEXOS_OFICIAL, 'C_Seleccion_Modelo_y_Congelamiento', '05_hiperparametros_finales.json'),
    os.path.join(RUTA_RESULTADOS, '05_hiperparametros_finales.json'),
]

def resolver_existente(candidatos, etiqueta):
    for p in candidatos:
        if os.path.exists(p):
            print(f'{etiqueta}: {p}')
            return p
    raise FileNotFoundError(f'No se encontró {etiqueta}. Candidatos: {candidatos}')

RUTA_DATASET = resolver_existente(DATASET_CANDIDATOS, 'Dataset congelado')
RUTA_PARAMS = resolver_existente(PARAMS_CANDIDATOS, 'Hiperparámetros congelados')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

SHA256_ESPERADO_DATASET = '6e14bbc167e969759e4bea4d794825e5fcd4bcce8781cd743e3f0f84f607c716'
SHA256_ESPERADO_PARAMS  = '35ab4669b1d5c30b619b12436363c54f0d70e1fba654b26ad927083a82ba88f9'

sha_dataset = sha256_file(RUTA_DATASET)
sha_params = sha256_file(RUTA_PARAMS)

print('SHA256 dataset:', sha_dataset)
print('SHA256 params :', sha_params)
print('Salida oficial:', RUTA_ANEXO)

assert sha_dataset == SHA256_ESPERADO_DATASET, 'HASH DATASET NO COINCIDE CON MANIFIESTO L'
assert sha_params == SHA256_ESPERADO_PARAMS, 'HASH PARÁMETROS NO COINCIDE CON MANIFIESTO L'


## 1. Carga de artefactos congelados y controles de entrada

In [ ]:
df = pd.read_csv(RUTA_DATASET)
with open(RUTA_PARAMS, 'r') as f:
    FINAL_PARAMS = json.load(f)

FEATURES_FINAL = [
    'Earthquake_Magnitude',
    'ClstD_km',
    'Vs30_Selected_for_Analysis_m_s'
]

assert len(df) == 1987
assert df['NGAsubEQID'].nunique() == 108
assert df['NGAsubSSN'].nunique() == 692
assert FINAL_PARAMS == {
    'subsample': 0.7,
    'n_estimators': 400,
    'min_samples_split': 15,
    'min_samples_leaf': 2,
    'max_depth': 2,
    'learning_rate': 0.05
}

# IMPORTANTE PARA REPRODUCIBILIDAD EXACTA:
# El LOEO original de V5.1 no se construyó filtrando simplemente Pisco
# y conservando el orden original de df. Primero se reconstruyeron las
# particiones development/test mediante GroupShuffleSplit(seed=42) y luego
# se concatenaron df_dev + df_test. Con subsample=0.7, el orden de las filas
# forma parte del estado computacional del GradientBoostingRegressor porque
# la selección pseudoaleatoria de observaciones depende de la posición.
pisco_mask = df['Earthquake_Name'].astype(str).str.contains('Pisco', case=False, na=False)
ids_pisco = df.loc[pisco_mask, 'NGAsubEQID'].dropna().unique()
assert len(ids_pisco) == 1
PISCO_EQID = ids_pisco[0]

df_pool = df[df['NGAsubEQID'] != PISCO_EQID].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)
idx_dev, idx_test = next(
    gss.split(df_pool, groups=df_pool['NGAsubEQID'])
)

df_dev = df_pool.iloc[idx_dev].copy().reset_index(drop=True)
df_test = df_pool.iloc[idx_test].copy().reset_index(drop=True)

assert set(df_dev['NGAsubEQID']).isdisjoint(set(df_test['NGAsubEQID']))
assert PISCO_EQID not in set(df_dev['NGAsubEQID'])
assert PISCO_EQID not in set(df_test['NGAsubEQID'])

# Este orden reproduce exactamente la celda LOEO del notebook maestro V5.1.
pool = pd.concat([df_dev, df_test], ignore_index=True)

assert len(df_dev) == 1583
assert len(df_test) == 382
assert len(pool) == 1965

assert int((df['NGAsubEQID'] == PISCO_EQID).sum()) == 22
assert df_dev['NGAsubEQID'].nunique() == 85
assert df_test['NGAsubEQID'].nunique() == 22
assert pool['NGAsubEQID'].nunique() == 107

print('Muestra final:', len(df), 'registros')
print('Pisco retenido:', int((df['NGAsubEQID'] == PISCO_EQID).sum()), 'registros')
print('Desarrollo:', len(df_dev), 'registros')
print('Test interno:', len(df_test), 'registros')
print('Pool LOEO:', len(pool), 'registros y', pool['NGAsubEQID'].nunique(), 'eventos')


## 2. Funciones de evaluación
El residual se define como `ln(Sa_pred) − ln(Sa_obs)`, en concordancia con la tesis.

In [ ]:
def metricas(y_true, y_pred):
    y_true=np.asarray(y_true,float)
    y_pred=np.asarray(y_pred,float)
    resid=y_pred-y_true
    r2=np.nan
    if len(y_true)>=2 and np.var(y_true)>0:
        r2=r2_score(y_true,y_pred)
    return {
        'R2': r2,
        'RMSE': np.sqrt(mean_squared_error(y_true,y_pred)),
        'MAE': mean_absolute_error(y_true,y_pred),
        'BIAS': resid.mean()
    }

def export_csv(x,nombre):
    path=os.path.join(RUTA_ANEXO,nombre)
    x.to_csv(path,index=False)
    print('Guardado:',path)
    return path

## 3. Reconstrucción LOEO estándar y LOEO con purga de estaciones

Para cada terremoto del pool no-Pisco:

1. se retira el evento completo;
2. se entrena Gradient Boosting V5.1 con los demás eventos y se predice el evento omitido;
3. en la sensibilidad con purga, además se eliminan del entrenamiento todas las estaciones presentes en el evento omitido;
4. cada registro recibe una única predicción fuera de evento.

Este procedimiento replica los bloques LOEO del notebook maestro congelado.

### Control crítico de orden de filas

La reconstrucción conserva deliberadamente el orden `df_dev → df_test` utilizado por el notebook maestro V5.1 antes del LOEO. Esto no es un detalle cosmético: el modelo congelado usa `subsample=0.7`, por lo que el muestreo pseudoaleatorio interno de Gradient Boosting depende del orden de las observaciones. Alterar ese orden cambia ligeramente las predicciones aunque se mantengan la misma semilla y los mismos hiperparámetros.

Por ello, una reproducción válida del análisis de dominio debe reconstruir primero la partición original mediante `GroupShuffleSplit(test_size=0.20, random_state=42)` y después concatenar `df_dev` y `df_test`, exactamente como en el notebook maestro V5.1.

In [ ]:
events = pool['NGAsubEQID'].unique()
y_pool = np.log(pool['T1pt000S'].to_numpy())
pred_loeo = np.full(len(pool), np.nan)
pred_purge = np.full(len(pool), np.nan)
n_train_purge = np.full(len(pool), np.nan)

for k, ev in enumerate(events, start=1):
    mask = pool['NGAsubEQID'].eq(ev).to_numpy()
    va = pool.loc[mask]
    tr = pool.loc[~mask]

    mod = GradientBoostingRegressor(random_state=SEED, **FINAL_PARAMS)
    mod.fit(tr[FEATURES_FINAL], np.log(tr['T1pt000S'].to_numpy()))
    pred_loeo[mask] = mod.predict(va[FEATURES_FINAL])

    stations = set(va['NGAsubSSN'])
    trp = pool.loc[(~mask) & (~pool['NGAsubSSN'].isin(stations))].copy()
    modp = GradientBoostingRegressor(random_state=SEED, **FINAL_PARAMS)
    modp.fit(trp[FEATURES_FINAL], np.log(trp['T1pt000S'].to_numpy()))
    pred_purge[mask] = modp.predict(va[FEATURES_FINAL])
    n_train_purge[mask] = len(trp)

    if k % 10 == 0 or k == len(events):
        print(f'{k}/{len(events)} eventos completados')

assert np.isfinite(pred_loeo).all()
assert np.isfinite(pred_purge).all()

m_global_loeo = metricas(y_pool, pred_loeo)
m_global_purge = metricas(y_pool, pred_purge)
print('LOEO global:', m_global_loeo)
print('LOEO + purga:', m_global_purge)

# Control contra cifras oficiales del notebook V5.1
assert round(m_global_loeo['R2'],4) == 0.8725
assert round(m_global_purge['R2'],4) == 0.8656

pred_reg = pool[[
    'NGAsubEQID','NGAsubSSN','Earthquake_Name',
    'Earthquake_Magnitude','ClstD_km','Vs30_Selected_for_Analysis_m_s','T1pt000S'
]].copy()
pred_reg['lnSa_obs'] = y_pool
pred_reg['lnSa_pred_LOEO'] = pred_loeo
pred_reg['lnSa_pred_LOEO_purga'] = pred_purge
pred_reg['residual_LOEO'] = pred_loeo-y_pool
pred_reg['residual_LOEO_purga'] = pred_purge-y_pool
pred_reg['N_train_purga_evento'] = n_train_purge

export_csv(pred_reg, 'F_01_LOEO_predicciones_por_registro.csv')

## 4. Diagnóstico específico de la ventana paramétrica de Lima
La tesis define Mw 7.5–8.5 y Rrup 50–150 km. El diagnóstico utiliza las predicciones LOEO ya generadas, por lo que no se recalibra V5.1 dentro de esa ventana.

In [ ]:
mask_lima = (
    (pred_reg['Earthquake_Magnitude'] >= 7.5) &
    (pred_reg['Earthquake_Magnitude'] <= 8.5) &
    (pred_reg['ClstD_km'] >= 50) &
    (pred_reg['ClstD_km'] <= 150)
)
lima = pred_reg.loc[mask_lima].copy()

m_lima = metricas(lima['lnSa_obs'], lima['lnSa_pred_LOEO'])
m_lima_purge = metricas(lima['lnSa_obs'], lima['lnSa_pred_LOEO_purga'])

res_lima = pd.DataFrame([
    {'Escenario':'LOEO', 'N_registros':len(lima), 'N_eventos':lima['NGAsubEQID'].nunique(), 'N_estaciones':lima['NGAsubSSN'].nunique(), **m_lima},
    {'Escenario':'LOEO + purga estaciones', 'N_registros':len(lima), 'N_eventos':lima['NGAsubEQID'].nunique(), 'N_estaciones':lima['NGAsubSSN'].nunique(), **m_lima_purge}
])
display(res_lima)

assert len(lima) == 99
assert lima['NGAsubEQID'].nunique() == 8
assert lima['NGAsubSSN'].nunique() == 87
assert round(m_lima['R2'],4) == 0.0079
assert round(m_lima['RMSE'],4) == 1.5710
assert round(m_lima['MAE'],4) == 1.0077
assert round(m_lima['BIAS'],4) == 0.3273
assert round(m_lima_purge['R2'],4) == -0.0066
assert round(m_lima_purge['RMSE'],4) == 1.5824
assert round(m_lima_purge['MAE'],4) == 1.0117
assert round(m_lima_purge['BIAS'],4) == 0.4337

export_csv(res_lima, 'F_02_metricas_ventana_Lima.csv')

## 5. Matriz refinada Mw × Rrup (64 celdas)

**Intervalos de Mw:** 5.0–<5.5, 5.5–<6.0, 6.0–<6.5, 6.5–<7.0, 7.0–<7.5, 7.5–<8.0, 8.0–<8.5 y 8.5–9.0.

**Intervalos de Rrup:** <25, 25–<50, 50–<75, 75–<100, 100–<150, 150–<200, 200–<300 y ≥300 km.

Clasificación post hoc descrita en la tesis:

- **favorable:** ≥5 eventos, R²_LOEO ≥0.50 y R²_purga ≥0.40;
- **limitado:** ≥3 eventos y ambos R² >0;
- **exploratorio:** desempeño nulo, negativo o inestable;
- **insuficiente:** <3 eventos.

In [ ]:
MW_EDGES=[5.0,5.5,6.0,6.5,7.0,7.5,8.0,8.5,9.000000001]
MW_LABELS=['5.0–<5.5','5.5–<6.0','6.0–<6.5','6.5–<7.0','7.0–<7.5','7.5–<8.0','8.0–<8.5','8.5–9.0']
RR_EDGES=[-np.inf,25,50,75,100,150,200,300,np.inf]
RR_LABELS=['<25','25–<50','50–<75','75–<100','100–<150','150–<200','200–<300','≥300']

pred_reg['Mw_bin'] = pd.cut(pred_reg['Earthquake_Magnitude'], bins=MW_EDGES, labels=MW_LABELS, right=False, include_lowest=True)
pred_reg['Rrup_bin'] = pd.cut(pred_reg['ClstD_km'], bins=RR_EDGES, labels=RR_LABELS, right=False, include_lowest=True)

rows=[]
for mwlab in MW_LABELS:
    for rrlab in RR_LABELS:
        g=pred_reg[(pred_reg['Mw_bin']==mwlab)&(pred_reg['Rrup_bin']==rrlab)].copy()
        n=len(g); ne=g['NGAsubEQID'].nunique(); ns=g['NGAsubSSN'].nunique()
        a=metricas(g['lnSa_obs'],g['lnSa_pred_LOEO']) if n>=2 else {'R2':np.nan,'RMSE':np.nan,'MAE':np.nan,'BIAS':np.nan}
        b=metricas(g['lnSa_obs'],g['lnSa_pred_LOEO_purga']) if n>=2 else {'R2':np.nan,'RMSE':np.nan,'MAE':np.nan,'BIAS':np.nan}

        if ne < 3:
            clase='insuficiente'
        elif ne >= 5 and a['R2'] >= 0.50 and b['R2'] >= 0.40:
            clase='favorable'
        elif a['R2'] > 0 and b['R2'] > 0:
            clase='limitado'
        else:
            clase='exploratorio'

        rows.append({
            'Mw_bin':mwlab,'Rrup_bin':rrlab,
            'N_registros':n,'N_eventos':ne,'N_estaciones':ns,
            'R2_LOEO':a['R2'],'RMSE_LOEO':a['RMSE'],'MAE_LOEO':a['MAE'],'BIAS_LOEO':a['BIAS'],
            'R2_purga':b['R2'],'RMSE_purga':b['RMSE'],'MAE_purga':b['MAE'],'BIAS_purga':b['BIAS'],
            'Clasificacion':clase
        })

matriz=pd.DataFrame(rows)
assert len(matriz)==64
resumen_clase=matriz['Clasificacion'].value_counts().reindex(['favorable','limitado','exploratorio','insuficiente'],fill_value=0)
print(resumen_clase)

assert resumen_clase.to_dict() == {'favorable':3,'limitado':12,'exploratorio':21,'insuficiente':28}

fav=matriz[matriz['Clasificacion']=='favorable'].copy()
display(fav[['Mw_bin','Rrup_bin','N_eventos','R2_LOEO','R2_purga']])

# Cifras publicadas de las tres celdas favorables
expected={
    ('6.0–<6.5','≥300'):(0.6808,0.6682),
    ('6.5–<7.0','≥300'):(0.7178,0.6941),
    ('7.0–<7.5','≥300'):(0.6271,0.6470),
}
for key,(r2a,r2b) in expected.items():
    row=matriz[(matriz['Mw_bin']==key[0])&(matriz['Rrup_bin']==key[1])].iloc[0]
    assert round(row['R2_LOEO'],4)==r2a
    assert round(row['R2_purga'],4)==r2b

export_csv(matriz,'F_03_matriz_64_celdas_Mw_Rrup.csv')
# Alias contractual consumido por 00_REGENERAR_FIGURAS_TESIS.ipynb
export_csv(matriz,'F_MATRIZ_DOMINIO_MW_RRUP_RECONSTRUIDA_OFICIAL.csv')
export_csv(resumen_clase.rename_axis('Clasificacion').reset_index(name='N_celdas'),'F_04_resumen_clasificacion_dominio.csv')


## 6. Regeneración de la Figura 12
La figura se genera directamente a partir de `F_03_matriz_64_celdas_Mw_Rrup.csv`. La clasificación es descriptiva y post hoc; no constituye un umbral normativo.

In [ ]:
class_code={'insuficiente':0,'exploratorio':1,'limitado':2,'favorable':3}
Z=np.full((len(MW_LABELS),len(RR_LABELS)),np.nan)
for i,mw in enumerate(MW_LABELS):
    for j,rr in enumerate(RR_LABELS):
        row=matriz[(matriz['Mw_bin']==mw)&(matriz['Rrup_bin']==rr)].iloc[0]
        Z[i,j]=class_code[row['Clasificacion']]

fig,ax=plt.subplots(figsize=(11,6))
im=ax.imshow(Z,aspect='auto',origin='lower',vmin=0,vmax=3,cmap='viridis')
ax.set_xticks(range(len(RR_LABELS)),RR_LABELS,rotation=35,ha='right')
ax.set_yticks(range(len(MW_LABELS)),MW_LABELS)
ax.set_xlabel('Rrup (km)')
ax.set_ylabel('Mw')
ax.set_title('Clasificación operativa del dominio de aplicabilidad de V5.1')
for i in range(len(MW_LABELS)):
    for j in range(len(RR_LABELS)):
        row=matriz[(matriz['Mw_bin']==MW_LABELS[i])&(matriz['Rrup_bin']==RR_LABELS[j])].iloc[0]
        ax.text(j,i,f"{row['Clasificacion']}\nE={int(row['N_eventos'])}",ha='center',va='center',fontsize=7)
fig.tight_layout()
ruta_fig=os.path.join(RUTA_ANEXO,'Figura_12_Dominio_Aplicabilidad_RECONSTRUIDA.png')
fig.savefig(ruta_fig,dpi=300,bbox_inches='tight')
plt.show()
print('Guardado:',ruta_fig)

## 7. Auditoría final
Una ejecución válida debe superar todos los `assert` anteriores. Si falla un control, los resultados no deben incorporarse al paquete definitivo de anexos.

In [ ]:
auditoria=pd.DataFrame([
    ['N dataset',len(df),1987],
    ['N eventos dataset',df['NGAsubEQID'].nunique(),108],
    ['N estaciones dataset',df['NGAsubSSN'].nunique(),692],
    ['R2 LOEO global',m_global_loeo['R2'],0.8725],
    ['R2 LOEO purga global',m_global_purge['R2'],0.8656],
    ['N ventana Lima',len(lima),99],
    ['R2 Lima LOEO',m_lima['R2'],0.0079],
    ['R2 Lima purga',m_lima_purge['R2'],-0.0066],
    ['Celdas favorables',int(resumen_clase['favorable']),3],
    ['Celdas limitadas',int(resumen_clase['limitado']),12],
    ['Celdas exploratorias',int(resumen_clase['exploratorio']),21],
    ['Celdas insuficientes',int(resumen_clase['insuficiente']),28],
],columns=['Control','Valor_obtenido','Valor_publicado_aprox'])
display(auditoria)
export_csv(auditoria,'F_05_auditoria_reconstruccion_dominio.csv')
print('AUDITORÍA ANEXO F: OK')

# Manifiesto SHA-256 de outputs oficiales del Anexo F.
# Esta celda solo puede alcanzarse si el entorno exacto y todos los asserts anteriores dieron PASS.
OUTPUTS_OFICIALES = [
    'F_01_LOEO_predicciones_por_registro.csv',
    'F_02_metricas_ventana_Lima.csv',
    'F_03_matriz_64_celdas_Mw_Rrup.csv',
    'F_MATRIZ_DOMINIO_MW_RRUP_RECONSTRUIDA_OFICIAL.csv',
    'F_04_resumen_clasificacion_dominio.csv',
    'F_05_auditoria_reconstruccion_dominio.csv',
    'Figura_12_Dominio_Aplicabilidad_RECONSTRUIDA.png',
]

manifest_rows = []
for nombre in OUTPUTS_OFICIALES:
    p = os.path.join(RUTA_ANEXO, nombre)
    assert os.path.exists(p), f'Falta output oficial esperado: {p}'
    manifest_rows.append({
        'archivo': nombre,
        'sha256': sha256_file(p),
        'bytes': os.path.getsize(p),
        'estado': 'OFICIAL_RECONSTRUIDO_CERTIFICADO'
    })

manifest_f = pd.DataFrame(manifest_rows)
export_csv(manifest_f, 'F_SHA256_OUTPUTS_OFICIALES.csv')
display(manifest_f)

print('AUDITORÍA ANEXO F: OK — entorno, entradas, resultados y hashes certificados')
